<a href="https://colab.research.google.com/github/AbhinavDharam/Netflix-Content-Strategy-Viewer-Preference-Analysis/blob/main/Netflix_Viewer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Content Strategy & Market Opportunity Recommendation System

This system transforms the Netflix titles dataset into an intelligent opportunity finder. It predicts show audiences using a trained machine learning model and computes opportunity scores based on catalog scarcity.

## 1. Import Libraries

In [ ]:
import pandas as pd  # For tabular data loading and manipulation
import numpy as np  # For numerical operations and matrix evaluations

from sklearn.model_selection import train_test_split  # For splitting data into training and testing sets
from sklearn.ensemble import RandomForestClassifier  # For supervised modeling of feature combinations
from sklearn.feature_extraction.text import TfidfVectorizer  # For converting plot text descriptions into numerical features
from sklearn.preprocessing import MultiLabelBinarizer  # For encoding multi-label genre lists into binary vectors
from sklearn.metrics import classification_report, accuracy_score  # For reporting evaluation metrics

## 2. Load Dataset

In [ ]:
# Load cleaned dataset
df = pd.read_csv("cleaned_netflix_data.csv")
df.head(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added
0,s1,TV Show,3%,Unknown,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,2020,8
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,2016-12-23,2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...,2016,12


## 3. Data Cleaning

In [ ]:
# Clean missing values
for col in ['listed_in', 'director', 'cast', 'rating', 'description', 'country']:
    df[col] = df[col].fillna('Unknown')

## 4. Feature Engineering

In [ ]:
# Feature Engineering: Map raw ratings into 3 Strategic Audience Tiers
rating_map = {
    'TV-MA': 'Adult', 'R': 'Adult', 'NC-17': 'Adult',
    'TV-14': 'Teen/General', 'PG-13': 'Teen/General', 'TV-PG': 'Teen/General',
    'TV-Y': 'Kids/Family', 'TV-Y7': 'Kids/Family', 'TV-G': 'Kids/Family', 'G': 'Kids/Family', 'PG': 'Kids/Family'
}
df['audience_tier'] = df['rating'].map(rating_map).fillna('Other')
df_filtered = df[df['audience_tier'] != 'Other'].copy()

# Parse comma-separated genre lists
df_filtered['genres_list'] = df_filtered['listed_in'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip() != ''])

# Categorical Encoding
mlb_genres = MultiLabelBinarizer()
genres_encoded = pd.DataFrame(
    mlb_genres.fit_transform(df_filtered['genres_list']),
    columns=[f"{c}" for c in mlb_genres.classes_],
    index=df_filtered.index
)

## 5. Audience Label Creation

In [ ]:
y = df_filtered['audience_tier']

## 6. TF-IDF Feature Extraction

In [ ]:
tfidf = TfidfVectorizer(max_features=300, stop_words='english')
tfidf_features = pd.DataFrame(
    tfidf.fit_transform(df_filtered['description']).toarray(),
    columns=[f"Text_{w}" for w in tfidf.get_feature_names_out()],
    index=df_filtered.index
)

## 7. Random Forest Training

In [ ]:
# Construct Feature Matrix
X = pd.concat([genres_encoded, tfidf_features], axis=1)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model Training
rf_model = RandomForestClassifier(n_estimators=150, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluation
y_pred = rf_model.predict(X_test)
print("=" * 60)
print(f"Strategic Audience Tier Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("=" * 60)
print("\nDetailed Performance Report:\n")
print(classification_report(y_test, y_pred))

Strategic Audience Tier Accuracy: 64.67%

Detailed Performance Report:

              precision    recall  f1-score   support

       Adult       0.65      0.68      0.67       707
 Kids/Family       0.86      0.79      0.83       206
Teen/General       0.58      0.56      0.57       624

    accuracy                           0.65      1537
   macro avg       0.70      0.68      0.69      1537
weighted avg       0.65      0.65      0.65      1537



## 8. Prediction Function

In [ ]:
def map_genres(genre_str):
    """
    Helper function to map raw genre input string to matched classes in mlb_genres.
    """
    genres_list = [x.strip() for x in genre_str.split(",") if x.strip() != ""]
    mapped = []
    for g in genres_list:
        for cls in mlb_genres.classes_:
            if g.lower() in cls.lower() or cls.lower() in g.lower():
                mapped.append(cls)
                break
    return mapped

def predict_audience(genre, description):
    """
    Predicts the audience tier (Kids/Family, Teen/General, Adult) based on the genre and description.
    """
    mapped = map_genres(genre)
    genre_vector = mlb_genres.transform([mapped])
    genre_df = pd.DataFrame(genre_vector, columns=[f"{c}" for c in mlb_genres.classes_])

    desc_vector = tfidf.transform([description]).toarray()
    desc_df = pd.DataFrame(desc_vector, columns=[f"Text_{w}" for w in tfidf.get_feature_names_out()])

    X_input = pd.concat([genre_df, desc_df], axis=1)
    prediction = rf_model.predict(X_input)[0]
    return prediction

## 9. Recommendation Engine

In [ ]:
def normalize_country(country_str):
    """
    Standardizes country inputs for accurate catalog matching.
    """
    if not country_str or country_str == 'Unknown':
        return 'Unknown'
    c_clean = country_str.strip().lower()
    if c_clean in ['usa', 'united states of america', 'u.s.a.', 'u.s.']:
        return 'United States'
    elif c_clean in ['uk', 'united kingdom', 'u.k.']:
        return 'United Kingdom'
    return country_str

def find_similar_titles(genre, predicted_audience, director=None, country=None):
    """
    Finds titles matching the criteria and computes overall catalog frequencies for opportunity analysis.
    """
    mapped_genres = map_genres(genre)
    if mapped_genres:
        genre_mask = pd.Series(False, index=df_filtered.index)
        for g in mapped_genres:
            genre_mask = genre_mask | df_filtered['listed_in'].str.contains(g, case=False, na=False)
    else:
        genre_mask = df_filtered['listed_in'].str.contains(genre, case=False, na=False)

    matching_mask = genre_mask & (df_filtered['audience_tier'] == predicted_audience)

    if director and director != 'Unknown':
        matching_mask = matching_mask & df_filtered['director'].str.contains(director, case=False, na=False)

    norm_country = normalize_country(country)
    if norm_country and norm_country != 'Unknown':
        matching_mask = matching_mask & df_filtered['country'].str.contains(norm_country, case=False, na=False)

    matching_titles = df_filtered[matching_mask].copy()
    total_titles = len(df_filtered)

    # Calculate statistics on the overall catalog using mapped genres union
    if mapped_genres:
        genre_union_mask = pd.Series(False, index=df_filtered.index)
        for g in mapped_genres:
            genre_union_mask = genre_union_mask | df_filtered['listed_in'].str.contains(g, case=False, na=False)
        genre_count = genre_union_mask.sum()
    else:
        genre_count = df_filtered['listed_in'].str.contains(genre, case=False, na=False).sum()

    genre_stats = {
        'count': int(genre_count),
        'percentage': float(genre_count / total_titles * 100) if total_titles > 0 else 0.0
    }

    if director and director != 'Unknown':
        dir_count = df_filtered['director'].str.contains(director, case=False, na=False).sum()
        dir_stats = {
            'count': int(dir_count),
            'percentage': float(dir_count / total_titles * 100) if total_titles > 0 else 0.0
        }
    else:
        dir_stats = None

    if norm_country and norm_country != 'Unknown':
        cntry_count = df_filtered['country'].str.contains(norm_country, case=False, na=False).sum()
        cntry_stats = {
            'count': int(cntry_count),
            'percentage': float(cntry_count / total_titles * 100) if total_titles > 0 else 0.0
        }
    else:
        cntry_stats = None

    return matching_titles, genre_stats, dir_stats, cntry_stats

## 10. Opportunity Score Calculator

In [ ]:
def calculate_opportunity_score(genre_stats, audience_stats, director_stats=None, country_stats=None):
    """
    Computes a normalized opportunity score (0-100) based on catalog scarcity.
    Higher scarcity = Higher opportunity.
    """
    scores = []

    genre_scarcity = 100.0 - genre_stats['percentage']
    scores.append(genre_scarcity)

    audience_scarcity = 100.0 - audience_stats['percentage']
    scores.append(audience_scarcity)

    if director_stats is not None:
        dir_scarcity = 100.0 - director_stats['percentage']
        scores.append(dir_scarcity)
    else:
        dir_scarcity = None

    if country_stats is not None:
        cntry_scarcity = 100.0 - country_stats['percentage']
        scores.append(cntry_scarcity)
    else:
        cntry_scarcity = None

    opportunity_score = sum(scores) / len(scores)

    if opportunity_score >= 80.0:
        competition_level = 'Low'
    elif opportunity_score >= 55.0:
        competition_level = 'Medium'
    else:
        competition_level = 'High'

    explanation = f"Genre scarcity is {genre_scarcity:.1f}% and predicted audience scarcity is {audience_scarcity:.1f}%."
    if dir_scarcity is not None:
        explanation += f" Director scarcity is {dir_scarcity:.1f}%."
    if cntry_scarcity is not None:
        explanation += f" Country scarcity is {cntry_scarcity:.1f}%."

    return round(opportunity_score, 1), competition_level, explanation

## 11. Final Pipeline Function

In [ ]:
def recommend_content_strategy(genre, description, director=None, country=None):
    """
    Main pipeline function that predicts the audience, queries similar titles,
    calculates catalog scarcity, and generates a structured content strategy recommendation.
    """
    pred_audience = predict_audience(genre, description)
    matching_titles, genre_stats, dir_stats, cntry_stats = find_similar_titles(
        genre, pred_audience, director, country
    )

    total_titles = len(df_filtered)
    aud_count = (df_filtered['audience_tier'] == pred_audience).sum()
    audience_stats = {
        'count': int(aud_count),
        'percentage': float(aud_count / total_titles * 100) if total_titles > 0 else 0.0
    }

    opportunity_score, competition_level, scarcity_explanation = calculate_opportunity_score(
        genre_stats, audience_stats, dir_stats, cntry_stats
    )

    similar_list = matching_titles['title'].head(3).tolist()
    genre_clean = genre.split(',')[0].strip()

    rec_text = f"{pred_audience}-oriented {genre_clean} content appears "
    if competition_level == "Low":
        rec_text += "underrepresented in the Netflix catalog.\nThis combination presents a relatively low-competition opportunity."
    elif competition_level == "Medium":
        rec_text += "moderately represented in the Netflix catalog.\nThis combination presents a balanced opportunity with moderate competition."
    else:
        rec_text += "highly represented in the Netflix catalog.\nThis combination presents a high-competition space with significant existing content."

    # Pretty print the final strategy result
    print("=" * 60)
    print("AI CONTENT STRATEGY & MARKET OPPORTUNITY REPORT")
    print("=" * 60)
    print(f"Predicted Audience: {pred_audience}")
    print(f"Opportunity Score: {opportunity_score}%")
    print(f"Competition Level: {competition_level}")
    print(f"Matching Catalog Titles: {len(matching_titles)}")
    print(f"Similar Titles: {', '.join(similar_list) if similar_list else 'None found'}")
    print("\nStrategic Recommendation:")
    print(rec_text)
    print("=" * 60)

    return {
        "predicted_audience": pred_audience,
        "opportunity_score": opportunity_score,
        "competition_level": competition_level,
        "matching_titles_count": len(matching_titles),
        "similar_titles": similar_list,
        "recommendation_text": rec_text
    }

## 12. Example Predictions

In [ ]:
# Example 1: High opportunity / low competition case (from prompt example)
strategy_1 = recommend_content_strategy(
    genre="Sci-Fi",
    description="A group of teenagers discovers an alien civilization beneath the ocean.",
    director="Christopher Nolan",
    country="USA"
)

# Example 2: Typical comedy example case
strategy_2 = recommend_content_strategy(
    genre="Comedies",
    description="A funny family gets into silly arguments during their summer road trip.",
    director="Unknown",
    country="United States"
)

AI CONTENT STRATEGY & MARKET OPPORTUNITY REPORT
Predicted Audience: Teen/General
Opportunity Score: 78.3%
Competition Level: Medium
Matching Catalog Titles: 1
Similar Titles: Inception

Strategic Recommendation:
Teen/General-oriented Sci-Fi content appears moderately represented in the Netflix catalog.
This combination presents a balanced opportunity with moderate competition.
AI CONTENT STRATEGY & MARKET OPPORTUNITY REPORT
Predicted Audience: Adult
Opportunity Score: 62.0%
Competition Level: Medium
Matching Catalog Titles: 332
Similar Titles: #blackAF, 21 & Over, 30 Minutes or Less

Strategic Recommendation:
Adult-oriented Comedies content appears moderately represented in the Netflix catalog.
This combination presents a balanced opportunity with moderate competition.
